<center><h1>Meyyappan_Shammu_HW3</h1></center>
<br>
<br>

Name: Shammu Meyyappan
<br>
Github Username: ShammuMeyyappan
<br>
USC ID: 4085790627

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) Download Data

Package imports

In [15]:
import numpy as np
import pandas as pd

Get the AReM Data Set

In [16]:
ACTIVITIES = ["bending1", "bending2", "cycling", "lying", "sitting", "standing", "walking"]
COLUMNS = ["time", "avg_rss12", "var_rss12", "avg_rss13", "var_rss13", "avg_rss23", "var_rss23"]
SERIES = COLUMNS[1:]

NUM_FILES = {"bending1": 7, "bending2": 6, "cycling": 15, "lying": 15,
             "sitting": 15, "standing": 15, "walking": 15}


def load_file(activity, num):
    path = f"../data/{activity}/dataset{num}.csv"
    return pd.read_csv(path, skiprows=5, header=None, names=COLUMNS,
                       sep=r"[,\s]+", engine="python", usecols=range(7))

load_file("bending1", 1).head()

,time,avg_rss12,var_rss12,avg_rss13,var_rss13,avg_rss23,var_rss23
0,0,39.25,0.43,22.75,0.43,33.75,1.3
1,250,39.25,0.43,23.00,0.00,33.00,0.0
2,500,39.25,0.43,23.25,0.43,33.00,0.0
3,750,39.50,0.50,23.00,0.71,33.00,0.0
4,1000,39.50,0.50,24.00,0.00,33.00,0.0


### (b) Test and Train Data

In [17]:
instances = []
for activity in ACTIVITIES:
    for num in range(1, NUM_FILES[activity] + 1):
        if activity in ["bending1", "bending2"]:
            test_count = 2
        else:
            test_count = 3

        if num <= test_count:
            split = "test"
        else:
            split = "train"

        instances.append((activity, num, split))

splits = []
for activity, num, split in instances:
    splits.append(split)

print("Total:", len(instances))
print("Train:", splits.count("train"))
print("Test: ", splits.count("test"))

Total: 88
Train: 69
Test:  19


### (c) Feature Extraction

#### i. Research

Time series classification often uses simple statistical features such as mean, median, min, max, standard deviation, skewness, and kurtosis.

Sources:  
https://brainly.com/question/31028265  
https://stats.stackexchange.com/questions/50807/features-for-time-series-classification

#### ii. Extraction

In [18]:
rows = []
for activity, num, split in instances:
    df = load_file(activity, num)
    row = {}
    for col in SERIES:
        x = df[col]
        row[f"min_{col}"] = x.min()
        row[f"max_{col}"] = x.max()
        row[f"mean_{col}"] = x.mean()
        row[f"median_{col}"] = x.median()
        row[f"std_{col}"] = x.std()
        row[f"1st_quart_{col}"] = x.quantile(0.25)
        row[f"3rd_quart_{col}"] = x.quantile(0.75)
    rows.append(row)

features = pd.DataFrame(rows)
features.index = range(1, len(features) + 1)
features.index.name = "Instance"
features

,min_avg_rss12,max_avg_rss12,mean_avg_rss12,median_avg_rss12,std_avg_rss12,1st_quart_avg_rss12,3rd_quart_avg_rss12,min_var_rss12,max_var_rss12,mean_var_rss12,...,std_avg_rss23,1st_quart_avg_rss23,3rd_quart_avg_rss23,min_var_rss23,max_var_rss23,mean_var_rss23,median_var_rss23,std_var_rss23,1st_quart_var_rss23,3rd_quart_var_rss23
Instance,,,,,,,,,,,,,,,,,,,,,
1,37.25,45.00,40.624792,40.50,1.476967,39.25,42.0000,0.0,1.30,0.358604,...,2.188449,33.0000,36.00,0.00,1.92,0.570583,0.43,0.582915,0.0000,1.3000
2,38.00,45.67,42.812812,42.50,1.435550,42.00,43.6700,0.0,1.22,0.372437,...,1.995255,32.0000,34.50,0.00,3.11,0.571083,0.43,0.601010,0.0000,1.3000
3,35.00,47.40,43.954500,44.33,1.558835,43.00,45.0000,0.0,1.70,0.426250,...,1.999604,35.3625,36.50,0.00,1.79,0.493292,0.43,0.513506,0.0000,0.9400
4,33.00,47.75,42.179812,43.50,3.670666,39.15,45.0000,0.0,3.00,0.696042,...,3.849448,30.4575,36.33,0.00,2.18,0.613521,0.50,0.524317,0.0000,1.0000
5,33.00,45.75,41.678063,41.75,2.243490,41.33,42.7500,0.0,2.83,0.535979,...,2.411026,28.4575,31.25,0.00,1.79,0.383292,0.43,0.389164,0.0000,0.5000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84,19.50,45.33,33.586875,34.25,4.650935,30.25,37.0000,0.0,14.67,4.576562,...,3.283983,13.7300,18.25,0.00,8.32,3.259729,3.11,1.640243,2.0500,4.3225
85,19.75,45.50,34.322750,35.25,4.752477,31.00,38.0000,0.0,13.47,4.456333,...,3.119856,13.5000,17.75,0.00,9.67,3.432563,3.20,1.732727,2.1575,4.5650
86,19.50,46.00,34.546229,35.25,4.842294,31.25,37.8125,0.0,12.47,4.371958,...,2.823124,14.0000,17.75,0.00,10.00,3.338125,3.08,1.656742,2.1600,4.3350


#### iii. Standard Deviation

In [19]:
boot_stds = []
for b in range(1000):
    sample = features.sample(n=len(features), replace=True, random_state=b)
    boot_stds.append(sample.std())
boot_stds = pd.DataFrame(boot_stds)

result = pd.DataFrame({
    "std": features.std(),
    "90% CI lower": boot_stds.quantile(0.05),
    "90% CI upper": boot_stds.quantile(0.95),
})
result.round(3)

,std,90% CI lower,90% CI upper
min_avg_rss12,9.570,8.287,10.754
max_avg_rss12,4.394,3.330,5.308
mean_avg_rss12,5.336,4.713,5.879
median_avg_rss12,5.440,4.801,5.996
std_avg_rss12,1.772,1.565,1.950
1st_quart_avg_rss12,6.154,5.570,6.671
3rd_quart_avg_rss12,5.139,4.310,5.838
min_var_rss12,0.000,0.000,0.000
max_var_rss12,5.063,4.626,5.388
mean_var_rss12,1.574,1.398,1.715


#### iv. Select Features

The three most important time-domain features are mean, as the typical signal level indicates the body's position, standard deviation, as the signal variance indicates whether or not the body is in motion, and max, as the maximum signal difference captures if the body was at one point standing even if it is not anymore.

## 2. ISLR 3.7.4

### (a) Linear Train

We would expect the cubic regression training RSS to be lower than the linear one, since the cubic model possesses the ability to produce a linear fit by setting β3 = β2 = 0, and anything beyond that would only serve to fit the training data better. 

### (b) Linear Test

We would expect the linear regression test RSS to be lower than the cubic one, since the cubic model is likely to overfit the linear relationship between X and Y, causing it to perform poorly on new test data.

### (c) Not Linear Train

We would expect the cubic regression training RSS to be lower than the linear one for the same reasons as part (a), since the cubic model can use a linear model as the baseline and only improve upon its fit to the training data from there, regardless of how far away from linear the true relationship is.

### (d) Not Linear Testing

There is not enough information to tell whether the cubic or linear regression test RSS would be lower in this case, since if the relationship is further from linear then the cubic model would perform better due to its increased ability to capture non-linear patterns, but if the relationship is closer to linear then the cubic model is likely to have overfit the training data and perform poorly with the test data.